# Phase 3 — Panel Regression (Fixed Effects)

**Goal:** Test whether nurse staffing is statistically significantly associated with patient outcomes, controlling for GDP, health expenditure, and hospital beds.

Method: Fixed Effects panel regression (within estimator) using `linearmodels`.

Model: `outcome_it = β × nurses_it + γ × controls_it + α_i + ε_it`
- `i` = country, `t` = year
- `α_i` = country fixed effect (absorbs time-invariant differences)

Outcomes tested:
1. 30-day AMI mortality
2. 30-day stroke mortality
3. Average length of stay

In [ ]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

PROCESSED = '../data/processed/'
OUTPUTS = '../data/outputs/'

df = pd.read_csv(PROCESSED + 'master_dataset.csv')
df = df.dropna(subset=['nurses_per_10k','gdp_per_capita','health_exp_pct_gdp','beds_per_1000'])
df = df.set_index(['country', 'year'])
print('Panel shape:', df.shape)

## 1. Model: Staffing → AMI Mortality

In [ ]:
ami_data = df.dropna(subset=['mortality_ami_30d'])

model_ami = PanelOLS(
    dependent=ami_data['mortality_ami_30d'],
    exog=sm.add_constant(ami_data[['nurses_per_10k','gdp_per_capita','health_exp_pct_gdp','beds_per_1000']]),
    entity_effects=True,
    time_effects=True
)
res_ami = model_ami.fit(cov_type='clustered', cluster_entity=True)
print(res_ami.summary)

## 2. Model: Staffing → Stroke Mortality

In [ ]:
stroke_data = df.dropna(subset=['mortality_stroke_30d'])

model_stroke = PanelOLS(
    dependent=stroke_data['mortality_stroke_30d'],
    exog=sm.add_constant(stroke_data[['nurses_per_10k','gdp_per_capita','health_exp_pct_gdp','beds_per_1000']]),
    entity_effects=True,
    time_effects=True
)
res_stroke = model_stroke.fit(cov_type='clustered', cluster_entity=True)
print(res_stroke.summary)

## 3. Model: Staffing → Length of Stay

In [ ]:
los_data = df.dropna(subset=['avg_length_of_stay'])

model_los = PanelOLS(
    dependent=los_data['avg_length_of_stay'],
    exog=sm.add_constant(los_data[['nurses_per_10k','gdp_per_capita','health_exp_pct_gdp','beds_per_1000']]),
    entity_effects=True,
    time_effects=True
)
res_los = model_los.fit(cov_type='clustered', cluster_entity=True)
print(res_los.summary)

## 4. Results Summary Table

In [ ]:
results = []
for name, res in [('AMI Mortality', res_ami), ('Stroke Mortality', res_stroke), ('Length of Stay', res_los)]:
    coef = res.params['nurses_per_10k']
    pval = res.pvalues['nurses_per_10k']
    ci_low, ci_high = res.conf_int().loc['nurses_per_10k']
    results.append({'Outcome': name, 'Coefficient': round(coef, 4), 'p-value': round(pval, 4), '95% CI Low': round(ci_low, 4), '95% CI High': round(ci_high, 4), 'Significant': pval < 0.05})

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUTS + 'regression_results.csv', index=False)
print(results_df.to_string(index=False))